# Suite EXP — Expressions

`Expressions.OfLiteral`, `OfOperation`, `OfVariable`, `OfLet`, `OfQuantifier` and `OfAny` follow the builder pattern
(`Data`, `Builder`, `Spec`, `resolve`) and have meta-schemas (`Schema`): ordinary value object schemas, registered as
`Expressions.*` in the dialect's `store`. An expression is a value of its kind's: `reader()` gives it, through which it
serializes, validates and compares like any value, and `DIALECT.term(value)` rebuilds an expression from one.
`Writer`s write expressions with methods. `Evaluators` computes them (suite EVL).

In [ ]:
import { Expressions as E } from "@mbse/expressions";
import * as Python from "@mbse/expressions/Dialects/Python";
import { Terms as F } from "@mbse/expressions/Framework";
import { Comparison as C, Errors, JSON, Plain, Proxies, Schemas as S, Validators, YAML } from "@mbse/schemas/Framework";
import { register } from "@mbse/expressions";

const store = register(new Proxies.OfStore()); // holds every dialect's meta-schemas
const A = F.ARGUMENT; // a value of any type: an expression, tagged with its kind's meta-schema's name

const { DecodeError, KeyError, ValueError } = Errors;
import { assert, equal, map, raises, same, text } from "./support.js";

const lit = new E.OfLiteral.Builder().value(3n).create();
const op = new E.OfOperation.Builder().name("f").arguments(1n, (a) => a.as_literal("x")).arguments(lit).create();
const values = (operation: E.OfOperation.Data) => operation.arguments.map((a) => (a as E.OfLiteral.Data).value);

## EXP-01 · Literals hold one native value

In [ ]:
assert(lit.value === 3n && lit.validate().length === 0 && lit.schema_name() === "Expressions.OfLiteral");
assert(E.OfLiteral.resolve(3n).value === 3n && E.OfLiteral.resolve(lit) === lit);
assert(E.OfLiteral.resolve((b: E.OfLiteral.Builder) => b.value("x")).value === "x");
for (const value of [0n, 2n ** 70n, -0, NaN, "", new Uint8Array(), false]) { // every native type
  assert(E.OfLiteral.resolve(value).validate().length === 0);
}
assert(same(new E.OfLiteral.Data().validate(), ["a literal needs a value"]));
assert(same(new E.OfLiteral.Data([1n]).validate(), ["a literal must hold a native value, got list"]));
// A builder keeps a value that is not native under `value`, for validation to report.
assert(same(new E.OfLiteral.Builder(new E.OfLiteral.Data([1n])).clone().value, [1n]));
assert(same(new E.OfLiteral.Builder().set("value", [2n] as never).create().value, [2n]));

## EXP-02 · Operations have a name and ordered arguments

In [ ]:
assert(op.name === "f" && same(values(op), [1n, "x", 3n]) && op.arguments[2] === lit);
assert(op.validate().length === 0 && op.schema_name() === "Expressions.OfOperation");
const nested = E.OfOperation.resolve((o: E.OfOperation.Builder) => o.name("not").arguments((a) => a.as_operation(op)));
assert(same(nested.arguments, [op]) && E.OfOperation.resolve(op) === op);
assert(same(new E.OfOperation.Data().validate(), ["an operation needs a name"]));
assert(same(new E.OfOperation.Data("").validate(), ["an operation needs a name"]));
assert(same(new E.OfOperation.Data(3n).validate(), ["an operation's name must be a str, got int"]));
const bad = new E.OfOperation.Data("f", [new E.OfLiteral.Data(), new E.OfOperation.Data(null, [new E.OfLiteral.Data([])]), 7n]);
assert(same(bad.validate(), [
  "argument 0: a literal needs a value",
  "argument 1: an operation needs a name",
  "argument 1: argument 0: a literal must hold a native value, got list",
  "argument 2: not an expression: 7",
]));

## EXP-03 · Cycles are reported, not followed; shared arguments are not cycles

In [ ]:
const loop = new E.OfOperation.Builder().name("f").create();
new E.OfOperation.Builder(loop).arguments(loop).update();
assert(same(loop.validate(), ["argument 0: the expression contains a cycle"]));
assert(new E.OfOperation.Builder().name("g").arguments(op, op).create().validate().length === 0);

## EXP-04 · Builders: create, clone and update; `OfAny` selects a kind; Spec errors

In [ ]:
const source = new E.OfOperation.Builder().name("f").arguments(1n).create();
const copy = new E.OfOperation.Builder(source).arguments(2n).clone();
assert(same(values(copy), [1n, 2n]) && source.arguments.length === 1);
assert(new E.OfOperation.Builder(source).name("g").update() === source && source.name === "g");
const literal = new E.OfLiteral.Builder().value(3n).create();
assert(new E.OfLiteral.Builder(literal).value(4n).clone().value === 4n && literal.value === 3n);
assert(new E.OfLiteral.Builder(literal).value(5n).update() === literal && literal.value === 5n);
raises(ValueError, () => new E.OfLiteral.Builder(literal).create(), "create() is only valid without a source instance");
raises(ValueError, () => new E.OfOperation.Builder().clone(), "clone() is only valid with a source instance");
raises(ValueError, () => new E.OfLiteral.Builder().update(), "update() is only valid with a source instance");
raises(TypeError, () => (new E.OfLiteral.Builder().create as any)(1));
raises(TypeError, () => new E.OfOperation.Builder(3n as any), "expected operation data to build from, got int");
raises(TypeError, () => new E.OfLiteral.Builder("x" as any), "expected literal data to build from, got str");

assert((new E.OfAny.Builder().as_literal(1n).create() as E.OfLiteral.Data).value === 1n);
assert((E.OfAny.resolve((a: E.OfAny.Builder) => a.as_operation((o) => o.name("f"))) as E.OfOperation.Data).name === "f");
assert((E.OfAny.resolve(2n) as E.OfLiteral.Data).value === 2n && E.OfAny.resolve(op) === op);
raises(ValueError, () => new E.OfAny.Builder(literal).create(), "create() is only valid without a source instance");
raises(ValueError, () => new E.OfAny.Builder().clone(), "clone() is only valid with a source instance");
raises(ValueError, () => new E.OfAny.Builder().update(), "update() is only valid with a source instance");
raises(ValueError, () => new E.OfAny.Builder().create(), "no kind selected");
const cloned = new E.OfAny.Builder(literal).clone() as E.OfLiteral.Data;
assert(cloned !== literal && cloned.value === literal.value);
const shallow = new E.OfAny.Builder(op).clone() as E.OfOperation.Data;
assert(shallow !== op && shallow.name === op.name && same(shallow.arguments, op.arguments));
assert((new E.OfAny.Builder(literal).as_literal(1n).clone() as E.OfLiteral.Data).value === 1n && literal.value === 5n);
assert(new E.OfAny.Builder(literal).as_literal(9n).update() === literal && literal.value === 9n);
const target = new E.OfOperation.Builder().name("f").create();
assert(new E.OfAny.Builder(target).as_operation((o) => o.name("h").arguments(1n)).update() === target);
assert(target.name === "h" && same(values(target), [1n]));
raises(TypeError, () => new E.OfAny.Builder(literal).as_operation(op).update(), "update() cannot change the kind of the source expression");

raises(TypeError, () => E.OfAny.resolve(BigInt), "a class is not a Spec here, got int");
raises(TypeError, () => E.OfAny.resolve(() => undefined), "a Spec callable must return its builder, got None");
raises(TypeError, () => E.OfOperation.resolve(3n), "expected an operation or a callable taking its builder, got 3");
raises(TypeError, () => E.OfAny.resolve([1n]), "expected an expression, a native value or a callable taking its builder, got [1]");
raises(TypeError, () => E.OfLiteral.resolve(null), "expected a native value, a literal or a callable taking its builder, got None");

## EXP-05 · The meta-schemas are ordinary value object schemas, registered in the dialect's store

In [ ]:
const kinds = [E.OfLiteral, E.OfOperation, E.OfVariable, E.OfLet, E.OfQuantifier] as const;
for (const schema of [...kinds.map((k) => k.Schema), E.OfAny.Schema]) {
  assert(schema.validate().length === 0);
}
assert(kinds.every((k) => !k.Schema.ref && k.Schema.adjacencies.size === 0)); // values, never linked
assert(same([...E.OfLiteral.Schema.properties.keys()], ["int", "float", "str", "bool", "bytes", "domain"]));
assert(same([...E.OfOperation.Schema.properties.keys()], ["name", "domain", "arguments"])); // its result's domain, for conversions
assert(same([...E.OfVariable.Schema.properties.keys()], ["name"]) && same([...E.OfLet.Schema.properties.keys()], ["name", "arguments"]));
assert(same([...E.OfQuantifier.Schema.properties.keys()], ["name", "quantifier", "arguments"]));
const argumentsSchema = E.OfOperation.Schema.properties.get("arguments")!.type; // a dense container of any dialect's expressions
assert(argumentsSchema instanceof S.OfContainer.Data && argumentsSchema.arrangement === "dense" && argumentsSchema.element === A);
assert(A instanceof S.OfAny.Value && A.types.length === 0);
assert(same(E.OfAny.Schema.branches.map((b) => b.type), kinds.map((k) => k.Schema)));
assert(same(kinds.map((k) => k.Data.KIND), ["literal", "operation", "variable", "let", "quantifier"]));
const names = [E.LITERAL, E.OPERATION, E.VARIABLE, E.LET, E.QUANTIFIER];
assert(same(names, ["Expressions.OfLiteral", "Expressions.OfOperation", "Expressions.OfVariable", "Expressions.OfLet",
  "Expressions.OfQuantifier"]));
assert(same(names.map((name) => store.registered(name)), kinds.map((k) => k.Schema))); // registered in any store, too
assert(same(names.map((name) => E.store.registered(name)), kinds.map((k) => k.Schema)));
assert(same(kinds.map((k) => new k.Data().schema_name()), names) && E.store === E.DIALECT.store);

## EXP-06 · Round trips through Plain, JSON and YAML rebuild `Data`; an expression is written nested

In [ ]:
const expr = new E.OfOperation.Builder().name("f").arguments(
  (a) => a.as_operation((o) => o.name("eq").arguments(1n, 1.5)), "s", new Uint8Array([1]), true).create();
const expected = JSON.ToJSON(E.store)(A, expr.reader());
const codecs: [(schema: any, value: any) => unknown, (store: any) => (schema: any, text: any) => unknown][] = [
  [Plain.ToPlain(E.store), Plain.FromPlain], [JSON.ToJSON(E.store), JSON.FromJSON], [YAML.ToYAML(E.store), YAML.FromYAML]];
for (const [encode, decode] of codecs) {
  const back = E.DIALECT.term(decode(E.store)(A, encode(A, expr.reader())) as any);
  assert(back instanceof E.OfOperation.Data && back !== expr && F.same(back, expr));
  assert(back.validate().length === 0 && JSON.ToJSON(E.store)(A, back.reader()) === expected);
}
const not_ = new E.OfOperation.Builder().name("not").arguments(true).create();
assert(JSON.ToJSON(E.store)(A, not_.reader()) ===
  '{"Expressions.OfOperation": {"name": "not", "arguments": [{"Expressions.OfLiteral": {"bool": true}}]}}');
assert(JSON.ToJSON(E.store)(E.OfOperation.Schema, not_.reader()) === // untagged, where its schema is given
  '{"name": "not", "arguments": [{"Expressions.OfLiteral": {"bool": true}}]}');
const alone = JSON.ToJSON(E.store)(E.OfLiteral.Schema, E.OfLiteral.resolve(-0).reader());
assert(alone === '{"float": -0.0}');
assert(Object.is((E.DIALECT.term(JSON.FromJSON(E.store)(E.OfLiteral.Schema, alone) as any) as E.OfLiteral.Data).value, -0));
for (const kind of kinds) { // nothing set: nothing written
  assert(JSON.ToJSON(E.store)(kind.Schema, new kind.Data().reader()) === "{}");
  assert(F.same(E.DIALECT.term(JSON.FromJSON(E.store)(kind.Schema, "{}") as any), new kind.Data()));
}
const gap = new E.OfLet.Data("x", null, E.OfLiteral.resolve(1n)); // an empty slot is null
assert(JSON.ToJSON(E.store)(E.OfLet.Schema, gap.reader()) ===
  '{"name": "x", "arguments": [null, {"Expressions.OfLiteral": {"int": 1}}]}');
assert(F.same(E.DIALECT.term(gap.reader()), gap) && (E.DIALECT.term(gap.reader()) as E.OfLet.Data).value === null);
raises(DecodeError, () => JSON.FromJSON(E.store)(E.OfLiteral.Schema, '{"kind": "operation"}'),
  "$.kind: the value object has no property 'kind'");
raises(DecodeError, () => JSON.FromJSON(E.store)(A, '{"Contact": {}}'), "no schema registered as 'Contact'");

## EXP-07 · An expression's value is built through the dialect's store, as any value is, and rebuilt as an expression

In [ ]:
const oneValue = (E.store.builder(E.LITERAL) as any).int(1n).create();
const built = (E.store.builder(E.OPERATION) as any).name("f").arguments(
  (p: any) => p.as_value((a: any) => a.as_container((c: any) => c.at(0n).as_object(oneValue) && c) || a)).create();
assert(F.same(E.DIALECT.term(built), E.operation("f", 1n).data) && (E.DIALECT.term(oneValue) as E.OfLiteral.Data).value === 1n);
assert(equal(Plain.ToPlain(E.store)(A, built), map({ "Expressions.OfOperation": { name: "f", arguments: [{ "Expressions.OfLiteral": { int: 1n } }] } })));
const Contact = new S.OfObject.Builder().name("Contact").properties(text("name")).create();
store.register(Contact);
raises(TypeError, () => E.DIALECT.term((store as any).Contact().name("Ada").create()), "not a value of a Basic kind: 'Contact'");

## EXP-08 · Validation and comparison work through the meta-schemas

In [ ]:
const validate = Validators.Validate(E.store);
assert(validate(E.OfOperation.Schema, expr.reader()).length === 0 && validate(E.OfLiteral.Schema, lit.reader()).length === 0);
assert(same(validate(E.OfLiteral.Schema, expr.reader()), [
  "the value is a 'Expressions.OfOperation', not an instance of the given schema"]));

function record(schema: S.OfObject.Data, expression: E.OfAny.Data): C.OfObject {
  return new C.OfObject(schema, expression.reader(), { value: true });
}

const [one, two] = [E.OfLiteral.resolve(1n), E.OfLiteral.resolve(2n)];
assert(record(E.OfLiteral.Schema, one).compare(record(E.OfLiteral.Schema, E.OfLiteral.resolve(1n))) === 0);
assert(record(E.OfLiteral.Schema, one).compare(record(E.OfLiteral.Schema, two)) === null); // objects are not ordered
assert(record(E.OfLiteral.Schema, one).compare(record(E.OfLiteral.Schema, E.OfLiteral.resolve(1.0))) === null);
const f1 = E.OfOperation.resolve((o: E.OfOperation.Builder) => o.name("f").arguments(1n));
const f2 = E.OfOperation.resolve((o: E.OfOperation.Builder) => o.name("f").arguments(2n));
assert(record(E.OfOperation.Schema, f1).compare(record(E.OfOperation.Schema, f2)) === null); // its arguments are its value
assert(record(E.OfOperation.Schema, f1).compare(record(E.OfOperation.Schema, E.operation("f", 1n).data as E.OfOperation.Data)) === 0);

## EXP-09 · An expression its meta-schema does not fit, or one with a cycle, has no value

In [ ]:
raises(DecodeError, () => new E.OfOperation.Data(3n).reader(), "$.name: expected str, got int");
raises(DecodeError, () => new E.OfOperation.Data([1n]).reader(), "$.name: expected str, got list");
raises(DecodeError, () => new E.OfOperation.Data("f", [new E.OfOperation.Data(3n)]).reader(),
  '$.arguments[0]["Expressions.OfOperation"].name: expected str, got int');
raises(TypeError, () => new E.OfLiteral.Data([1n]).reader(), "a literal must hold a native value, got list");
raises(ValueError, () => loop.reader(), "the expression contains a cycle");
const shared = E.literal(1n).data;
assert(JSON.ToJSON(E.store)(E.OfOperation.Schema, (E.operation("f", shared, shared).data as E.OfOperation.Data).reader()) ===
  '{"name": "f", "arguments": [{"Expressions.OfLiteral": {"int": 1}}, {"Expressions.OfLiteral": {"int": 1}}]}');

## EXP-10 · Variables and lets

In [ ]:
const v = new E.OfVariable.Builder().name("x").create();
assert(v.name === "x" && v.schema_name() === "Expressions.OfVariable" && E.OfVariable.resolve("x").name === "x");
assert(same(v.validate(), ["variable 'x' is not bound"]) && v.validate({ bound: ["x"] }).length === 0);
assert(same(new E.OfVariable.Data().validate(), ["a variable needs a name"]));
assert(same(new E.OfVariable.Data(1n).validate(), ["a variable's name must be a str, got int"]));
assert(E.OfVariable.resolve(v) === v && new E.OfVariable.Builder(v).name("y").clone().name === "y");
const let_ = new E.OfLet.Builder().name("x").value(1n).body(v).create();
assert(let_.name === "x" && (let_.value as E.OfLiteral.Data).value === 1n && let_.body === v && let_.validate().length === 0);
assert((new E.OfLet.Builder(let_).body(2n).clone().body as E.OfLiteral.Data).value === 2n && let_.body === v);
assert(new E.OfLet.Builder(let_).value(5n).update() === let_ && (let_.value as E.OfLiteral.Data).value === 5n && let_.body === v);
assert(E.OfLet.resolve((b: E.OfLet.Builder) => b.name("y").value(1n).body(2n)).name === "y" && E.OfLet.resolve(let_) === let_);
assert(same(new E.OfLet.Data("x").validate(), ["a let needs a value", "a let needs a body"]));
assert(same(new E.OfLet.Data(null, E.OfLiteral.resolve(1n), E.OfVariable.resolve("x")).validate(), [
  "a let needs a name", "body: variable 'x' is not bound"]));
const sameName = new E.OfLet.Data("y", E.OfVariable.resolve("y"), E.OfVariable.resolve("y"));
assert(same(sameName.validate(), ["value: variable 'y' is not bound"])); // the name is bound in the body only
assert((E.OfAny.resolve((a: E.OfAny.Builder) => a.as_variable("x")) as E.OfVariable.Data).name === "x");
assert(E.OfAny.resolve((a: E.OfAny.Builder) => a.as_let(let_)) === let_);
assert((new E.OfAny.Builder(v).clone() as E.OfVariable.Data).name === "x" && (new E.OfAny.Builder(let_).clone() as E.OfLet.Data).body === v);
const unnamed = [new E.OfOperation.Builder().create(), new E.OfVariable.Builder().create(), new E.OfLet.Builder().create()];
assert(same(unnamed.map((u) => u.name), [null, null, null]) && (unnamed[2] as E.OfLet.Data).value === null);
const openLet = new E.OfLet.Builder(new E.OfLet.Data("x")).clone();
assert(openLet.value === null && openLet.body === null);
raises(TypeError, () => E.OfLet.resolve(3n), "expected a let or a callable taking its builder, got 3");
raises(TypeError, () => E.OfVariable.resolve(3n), "expected a name, a variable or a callable taking its builder, got 3");
raises(TypeError, () => (new E.OfVariable.Builder() as any).arguments(1n), "a variable has no variadic arguments");
raises(KeyError, () => new E.OfLet.Builder().argument("other", 1n), "a let has no argument 'other'");
raises(KeyError, () => new E.OfVariable.Builder().set("value", 1n), "a variable has no attribute 'value'");

## EXP-11 · Writers build expressions with methods

In [ ]:
const self = E.variable("this");
const writer = self.age.ge(18n);
assert(writer.data instanceof E.OfOperation.Data && writer.data.name === "ge" && (writer.data.arguments[1] as E.OfLiteral.Data).value === 18n);
const got = (writer.data as E.OfOperation.Data).arguments[0] as E.OfOperation.Data;
assert(got.name === "get" && got.arguments[0] === self.data && (got.arguments[1] as E.OfLiteral.Data).value === "age");
assert(((self.get("eq").data as E.OfOperation.Data).arguments[1] as E.OfLiteral.Data).value === "eq"); // a property named like a method
const methods = [self.eq(1n), self.ne(1n), self.lt(1n), self.le(1n), self.gt(1n), self.ge(1n), self.and_(true), self.or_(true),
  self.not_(), self.implies(true), self.add(1n), self.sub(1n), self.mul(1n), self.neg(), self.has("x"), self.get("x")];
assert(same(methods.map((m) => (m.data as E.OfOperation.Data).name), ["eq", "ne", "lt", "le", "gt", "ge", "and", "or", "not",
  "implies", "add", "sub", "mul", "neg", "has", "get"]));
assert((E.literal(3n).data as E.OfLiteral.Data).value === 3n && (E.operation("in", 1n, 2n).data as E.OfOperation.Data).name === "in");
const bound = E.let_("a", 1n, E.variable("a"));
assert((bound.data as E.OfLet.Data).name === "a" && E.OfLet.resolve(bound) === bound.data && E.OfOperation.resolve(writer) === writer.data);
assert(E.OfAny.resolve(writer) === writer.data && E.OfAny.resolve((a: E.OfAny.Builder) => a.as_operation(writer)) === writer.data);
raises(TypeError, () => E.OfOperation.resolve(self), "expected an operation or a callable taking its builder"); // a variable, not an operation
assert(self.then === undefined && self.toJSON === undefined && self[Symbol.iterator] === undefined); // JavaScript's own probes are not properties

## EXP-12 · `Python.Text.FromFunction` reads an expression from a Python function (Python only)

In [ ]:
// Python only: a JavaScript function has no Python source to read, so TypeScript's Python.Text has no FromFunction.
assert(!("FromFunction" in Python.Text));